# 모델 2 — 정비 필요 시점 예측 (실제 비행기록)

NGAFID 실제 비행기록(28,935편)으로 "k일 내 정비가 필요해질 확률"의 생존곡선과 정비 필요 경보를 출력한다.

- 실행: GPU 런타임, 셀을 위에서 아래로 실행 (데이터 4.3GB 자동 다운로드, 최초 전처리 20~40분)
- 산출물: `results_ngafid_surv_v1.json`(지표), `preds_ngafid_surv_v1.npz`(예측), `fusion_interface_model2.json`(모델 1 연동 파라미터)

In [ ]:
# 설정 및 함수 정의
import os
DATA_ROOT = '/content/all_flights'
GDRIVE_TARS = ['/content/drive/MyDrive/제주항공/all_flight.tar.gz',
               '/content/drive/MyDrive/제주항공/all_flights.tar.gz']
GDOWN_ID = '1-0pVPhwRQoifT_VuQyGDLXuzYPYySX-Y'
CACHE = '/content/ngafid_cache.npz'
K_DAYS = 5
T_STEPS = 1024
K_FOLDS = 5; ENSEMBLE_M = 3; EPOCHS = 15; BATCH = 64; LR = 1e-3; WD = 1e-2
HYPERS2 = [(0.05, 1e-2, 1.0e-3), (0.10, 1e-2, 7.0e-4), (0.15, 3e-3, 1.3e-3)]
MC_DROPOUT_K = 4
CAL_FRAC = 0.15
LAMBDA_PMF = 1.0; LAMBDA_BIN = 1.0; USE_GRP = False; LAMBDA_GRP = 0.2
RECAL_KNOTS2 = 5
EXCLUDE_SAME = True
EVENT_GROUPING = 'event'
MIN_FLIGHT_LEN = 60
SEED0 = 0
SENSORS = ['volt1', 'volt2', 'amp1', 'amp2', 'FQtyL', 'FQtyR',
           'E1 FFlow', 'E1 OilT', 'E1 OilP', 'E1 RPM',
           'E1 CHT1', 'E1 CHT2', 'E1 CHT3', 'E1 CHT4',
           'E1 EGT1', 'E1 EGT2', 'E1 EGT3', 'E1 EGT4',
           'OAT', 'IAS', 'VSpd', 'NormAc', 'AltMSL']
EGT_IDX = [14, 15, 16, 17]; CHT_IDX = [10, 11, 12, 13]
NCH = len(SENSORS) + 2

import numpy as np, json, time, glob
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as Fn
from torch.utils.data import TensorDataset, DataLoader
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
rng = np.random.RandomState(0)

ZENODO_URL = 'https://zenodo.org/records/6624956/files/all_flight.tar.gz?download=1'

def _download_zenodo(tar):
    import urllib.request
    done = os.path.getsize(tar) if os.path.exists(tar) else 0
    req = urllib.request.Request(ZENODO_URL, headers={'Range': f'bytes={done}-'} if done else {})
    with urllib.request.urlopen(req, timeout=60) as r, open(tar, 'ab' if done else 'wb') as f:
        total = done + int(r.headers.get('Content-Length', 0)); t0 = time.time()
        while True:
            chunk = r.read(1 << 22)
            if not chunk: break
            f.write(chunk); done += len(chunk)
            if done % (1 << 28) < (1 << 22):
                print(f'    {done / 1e9:.2f}/{total / 1e9:.2f} GB [{time.time() - t0:.0f}s]')
    return os.path.getsize(tar) > 4e9

def ensure_data():
    if os.path.exists(os.path.join(DATA_ROOT, 'flight_header.csv')): return
    os.makedirs('/content', exist_ok=True)
    tar = '/content/all_flight.tar.gz'
    ok = os.path.exists(tar) and os.path.getsize(tar) > 4e9
    if not ok:
        for gt in GDRIVE_TARS:
            if os.path.exists(gt):
                print(f'  [데이터] Drive에서 tar 복사: {os.path.basename(gt)}...')
                import shutil; shutil.copy(gt, tar); ok = os.path.getsize(tar) > 4e9; break
    if not ok:
        try:
            print('  [데이터] gdown 시도(4.3GB — Drive 쿼터 초과 시 실패 정상)...')
            import subprocess; subprocess.run(['pip', 'install', '-q', 'gdown'], check=False)
            import gdown; gdown.download(id=GDOWN_ID, output=tar, quiet=False)
            ok = os.path.exists(tar) and os.path.getsize(tar) > 4e9
        except Exception as e:
            print(f'    gdown 실패({type(e).__name__}) → Zenodo 정식 경로로 전환')
    if not ok:
        print('  [데이터] Zenodo 다운로드(정식 원본, 재개 지원)...')
        try: ok = _download_zenodo(tar)
        except Exception as e: print(f'    Zenodo 실패: {e}')
    if not ok:
        raise RuntimeError(
            '자동 다운로드 실패. 수동 경로: (1) https://zenodo.org/records/6624956 에서 all_flight.tar.gz(4.3GB)를 받아 '
            'Drive의 제주항공 폴더에 업로드(파일명 유지) 후 재실행, 또는 (2) Kaggle '
            '"aviation-maintenance-dataset-from-the-ngafid"에서 다운로드. Drive에 있으면 이 셀이 자동 인식합니다.')
    print('  [데이터] 압축 해제...')
    import tarfile
    with tarfile.open(tar) as f: f.extractall('/content')
    assert os.path.exists(os.path.join(DATA_ROOT, 'flight_header.csv')), '해제 후 flight_header.csv 없음'

def build_labels(hdr):
    ba = hdr['before_after'].astype(str).values
    dd = hdr['date_diff'].fillna(0).astype(int).values
    lab = hdr['label'].astype(str).values
    is_b = ba == 'before'; is_a = ba == 'after'; is_s = ba == 'same'
    t = np.where(is_b, -dd, 10 ** 6)
    bins = np.full(len(hdr), K_DAYS, np.int64)
    bins[is_b & (t <= K_DAYS - 1)] = t[is_b & (t <= K_DAYS - 1)]
    keep = ~is_s if EXCLUDE_SAME else np.ones(len(hdr), bool)

    chg = np.concatenate([[True], lab[1:] != lab[:-1]])
    if EVENT_GROUPING == 'issue':
        event_id = np.cumsum(chg) - 1
    else:
        new_ev = chg.copy(); seen_end = False
        for i in range(len(hdr)):
            if chg[i]: seen_end = False
            if is_s[i] or is_a[i]: seen_end = True
            elif is_b[i] and seen_end: new_ev[i] = True; seen_end = False
        event_id = np.cumsum(new_ev) - 1

    bin2d_mask = (is_b & (t <= 2)) | (is_a & (np.abs(dd) <= 2))
    bin2d_label = is_b.astype(np.float32)
    return bins, is_b, is_a, t, event_id, keep, bin2d_label, bin2d_mask

def resample_flight(df_flight, t_steps=T_STEPS):
    d = df_flight.sort_values('timestep')
    X = d[SENSORS].astype(np.float64)
    X = X.ffill().bfill()
    arr = X.values
    L = len(arr)
    if L < 2: return None
    imb = np.stack([np.nanstd(arr[:, EGT_IDX], 1), np.nanstd(arr[:, CHT_IDX], 1)], 1)
    arr = np.concatenate([arr, imb], 1)
    xp = np.linspace(0, L - 1, t_steps)
    out = np.empty((t_steps, NCH), np.float32)
    base = np.arange(L, dtype=np.float64)
    for c in range(NCH):
        col = arr[:, c]
        if np.isnan(col).all(): out[:, c] = 0.0
        else: out[:, c] = np.interp(xp, base, col)
    return out

def extract_all(parq_dir, hdr):
    import pyarrow.parquet as pq
    files = sorted(glob.glob(os.path.join(parq_dir, '*.parquet'))) or sorted(glob.glob(os.path.join(parq_dir, 'part*')))
    assert files, f'parquet 파일 없음: {parq_dir}'
    want = set(hdr.index.tolist())
    N = len(hdr); X = np.zeros((N, T_STEPS, NCH), np.float32); got = np.zeros(N, bool)
    pos = {mi: i for i, mi in enumerate(hdr.index.tolist())}
    carry = None; t0 = time.time()
    for fi, f in enumerate(files):
        df = pq.read_table(f).to_pandas()
        if df.index.name != 'Master Index':
            if 'Master Index' in df.columns: df = df.set_index('Master Index')
        if carry is not None:
            df = pd.concat([carry, df]); carry = None
        idxs = df.index.values
        last_mi = idxs[-1]
        carry = df.loc[[last_mi]]
        body = df.loc[idxs != last_mi] if (idxs != last_mi).any() else df.iloc[0:0]
        for mi, g in body.groupby(level=0):
            if mi in want and not got[pos[mi]] and len(g) >= MIN_FLIGHT_LEN:
                r = resample_flight(g)
                if r is not None: X[pos[mi]] = r; got[pos[mi]] = True
        if fi % 50 == 0: print(f'    파티션 {fi + 1}/{len(files)} [{time.time() - t0:.0f}s]')
    if carry is not None and len(carry) > 0:
        for mi, g in carry.groupby(level=0):
            if mi in want and not got[pos[mi]] and len(g) >= MIN_FLIGHT_LEN:
                r = resample_flight(g)
                if r is not None: X[pos[mi]] = r; got[pos[mi]] = True
    return X, got

def load_all():
    ensure_data()
    hdr = pd.read_csv(os.path.join(DATA_ROOT, 'flight_header.csv'), index_col='Master Index')
    bins, is_b, is_a, t, eid, keep, b2l, b2m = build_labels(hdr)
    if os.path.exists(CACHE):
        z = np.load(CACHE); X, got = z['X'], z['got']
        print(f'  [캐시] {CACHE} 로드 ({got.sum()}편)')
    else:
        X, got = extract_all(os.path.join(DATA_ROOT, 'one_parq'), hdr)
        np.savez(CACHE, X=X, got=got); print(f'  [캐시] 저장 ({got.sum()}편)')
    m = keep & got
    print(f'  [데이터] 전체 {len(hdr)} → 사용 {m.sum()}편 (same 제외 {(~keep).sum()}, 추출실패 {(~got).sum()})')
    print(f'  [라벨] before {is_b[m].sum()} (빈분포 {np.bincount(bins[m][is_b[m]], minlength=K_DAYS+1).tolist()}) · after(절단) {is_a[m].sum()} · 이벤트 {len(set(eid[m].tolist()))}')
    return X[m], bins[m], is_b[m], is_a[m], t[m], eid[m], b2l[m], b2m[m]

class IncBlock(nn.Module):
    def __init__(self, cin, nf=32, p=0.1):
        super().__init__()
        self.bott = nn.Conv1d(cin, nf, 1)
        self.c1 = nn.Conv1d(nf, nf, 9, padding=4)
        self.c2 = nn.Conv1d(nf, nf, 19, padding=9)
        self.c3 = nn.Conv1d(nf, nf, 39, padding=19)
        self.mp = nn.Sequential(nn.MaxPool1d(3, 1, 1), nn.Conv1d(cin, nf, 1))
        self.bn = nn.BatchNorm1d(nf * 4); self.dp = nn.Dropout(p)
    def forward(self, x):
        b = self.bott(x)
        z = torch.cat([self.c1(b), self.c2(b), self.c3(b), self.mp(x)], 1)
        return self.dp(Fn.relu(self.bn(z)))

class SurvNet(nn.Module):
    def __init__(self, cin=NCH, nf=32, p=0.1, n_grp=5):
        super().__init__()
        self.blocks = nn.Sequential(IncBlock(cin, nf, p), nn.MaxPool1d(4),
                                    IncBlock(nf * 4, nf, p), nn.MaxPool1d(4),
                                    IncBlock(nf * 4, nf, p), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(nf * 4, 64), nn.ReLU(), nn.Dropout(max(p, 0.1) * 2))
        self.pmf = nn.Linear(64, K_DAYS + 1)
        self.bin = nn.Linear(64, 1)
        self.grp = nn.Linear(64, n_grp)
    def forward(self, x):
        z = self.head(self.blocks(x.transpose(1, 2)))
        return {'pmf': self.pmf(z), 'bin': self.bin(z).squeeze(-1), 'grp': self.grp(z)}

def _loader(arrs, sh=False):
    return DataLoader(TensorDataset(*[torch.tensor(a) for a in arrs]), batch_size=BATCH, shuffle=sh)

def train_one2(seed, Xtr, Btr, BLtr, BMtr, hyper):
    p_drop, wd_h, lr_h = hyper
    torch.manual_seed(seed); np.random.seed(seed)
    model = SurvNet(p=p_drop).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=lr_h, weight_decay=wd_h)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    ar = torch.arange(K_DAYS, device=dev)
    for ep in range(EPOCHS):
        model.train()
        for xb, bb, blb, bmb in _loader((Xtr, Btr, BLtr, BMtr), sh=True):
            xb, bb, blb, bmb = xb.to(dev), bb.to(dev), blb.to(dev), bmb.to(dev)
            opt.zero_grad(); o = model(xb)
            p = torch.softmax(o['pmf'], 1)
            F_ = p[:, :K_DAYS].cumsum(1)
            tgt = (ar.unsqueeze(0) >= bb.unsqueeze(1)).float()
            loss = LAMBDA_PMF * ((F_ - tgt) ** 2).sum(1).mean()
            loss = loss + LAMBDA_BIN * Fn.binary_cross_entropy_with_logits(o['bin'], blb)
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        sch.step()
    return model

def _set_mc2(model, on):
    for m in model.modules():
        if isinstance(m, nn.Dropout): m.train(on)

def _predict2(model, Xe):
    model.eval(); K = max(int(MC_DROPOUT_K), 1)
    out = {'pmf': [], 'bin': []}
    with torch.no_grad():
        for (xb,) in _loader((Xe,)):
            xd = xb.to(dev)
            _set_mc2(model, False); o0 = model(xd)
            out['bin'].append(torch.sigmoid(o0['bin']).cpu().numpy())
            acc = 0
            if K > 1: _set_mc2(model, True)
            for _ in range(K):
                o = o0 if K == 1 else model(xd)
                acc = acc + torch.softmax(o['pmf'], 1) / K
            out['pmf'].append(acc.cpu().numpy()); _set_mc2(model, False)
    return {k: np.concatenate(v) for k, v in out.items()}

def ensemble2(models, Xe):
    P = [_predict2(m, Xe) for m in models]
    return dict(pmf=np.stack([p['pmf'] for p in P]).mean(0),
                bin=np.stack([p['bin'] for p in P]).mean(0))

def rps_censored(pmf, bins):
    F_ = np.cumsum(pmf[:, :K_DAYS], 1)
    tgt = (np.arange(K_DAYS)[None, :] >= bins[:, None]).astype(np.float64)
    return float(((F_ - tgt) ** 2).sum(1).mean())

def fit_recal2(pmf_cal, bins_cal):
    F_ = np.cumsum(pmf_cal[:, :K_DAYS], 1)
    n = len(pmf_cal); taus = np.linspace(0, 1, RECAL_KNOTS2 + 2); R = np.zeros_like(taus)
    ev = bins_cal <= K_DAYS - 1
    Fy = np.where(ev, F_[np.arange(n), np.minimum(bins_cal, K_DAYS - 1)], F_[:, -1])
    for j, tv in enumerate(taus):
        a = float((Fy[ev] <= tv).sum())
        cn = ~ev
        if cn.any(): a += float(np.clip((tv - Fy[cn]) / np.maximum(1 - Fy[cn], 1e-9), 0, 1).sum())
        R[j] = a / max(n, 1)
    R[0] = 0.0; R[-1] = 1.0
    return taus, np.maximum.accumulate(R)

def apply_recal2(pmf, taus, R):
    F_ = np.cumsum(pmf[:, :K_DAYS], 1)
    F2 = np.interp(F_, taus, R)
    p = np.diff(np.concatenate([np.zeros((len(pmf), 1)), F2], 1), axis=1)
    out = np.concatenate([p, np.maximum(1 - F2[:, -1:], 0)], 1)
    out = np.maximum(out, 0); return out / np.maximum(out.sum(1, keepdims=True), 1e-12)

def auc_of(score, y):
    o = np.argsort(score); r = np.empty(len(score)); r[o] = np.arange(1, len(score) + 1)
    n1 = y.sum(); n0 = len(y) - n1
    if n1 == 0 or n0 == 0: return float('nan')
    return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

def f1_threshold(score, y):
    best_t, best_f1 = 0.5, -1
    for tt in np.quantile(score, np.linspace(0.05, 0.95, 37)):
        pr = score >= tt
        tp = (pr & (y == 1)).sum(); fp = (pr & (y == 0)).sum(); fn = ((~pr) & (y == 1)).sum()
        f1 = 2 * tp / max(2 * tp + fp + fn, 1)
        if f1 > best_f1: best_f1, best_t = f1, float(tt)
    return best_t

def run2():
    t0 = time.time()
    X, bins, is_b, is_a, t_days, eid, b2l, b2m = load_all()
    events = np.array(sorted(set(eid.tolist()))); rng.shuffle(events)
    folds = np.array_split(events, K_FOLDS)
    print(f'[학습] {len(X)}편 · 이벤트 {len(events)} → {K_FOLDS}-fold · M{ENSEMBLE_M} · MC(PMF만) K{MC_DROPOUT_K}')
    O = {k: [] for k in ['pmf', 'bin', 'bins', 'isb', 'isa', 't', 'eid', 'b2l', 'b2m', 'fold', 'thr']}
    for k, te_ev in enumerate(folds):
        te = np.isin(eid, te_ev); tr_ev = events[~np.isin(events, te_ev)]
        r2 = np.random.RandomState(100 + k); perm = r2.permutation(tr_ev)
        n_cal = max(5, int(round(CAL_FRAC * len(tr_ev))))
        cal_ev, fit_ev = perm[:n_cal], perm[n_cal:]
        trm = np.isin(eid, fit_ev); cam = np.isin(eid, cal_ev)
        mu = X[trm].reshape(-1, NCH).mean(0); sd = X[trm].reshape(-1, NCH).std(0) + 1e-6
        nm = lambda A: ((A - mu) / sd).astype(np.float32)
        models = [train_one2(SEED0 + 10 * k + i, nm(X[trm]), bins[trm], b2l[trm], b2m[trm], HYPERS2[i % len(HYPERS2)])
                  for i in range(ENSEMBLE_M)]
        E = ensemble2(models, nm(X[te])); Eca = ensemble2(models, nm(X[cam]))
        tks, Rv = fit_recal2(Eca['pmf'], bins[cam])
        pmf_c = apply_recal2(E['pmf'], tks, Rv)
        thr = f1_threshold(Eca['bin'][b2m[cam]], b2l[cam][b2m[cam]]) if b2m[cam].sum() > 20 else 0.5
        for key, val in [('pmf', pmf_c), ('bin', E['bin']), ('bins', bins[te]), ('isb', is_b[te]), ('isa', is_a[te]),
                         ('t', t_days[te]), ('eid', eid[te]), ('b2l', b2l[te]), ('b2m', b2m[te]),
                         ('fold', np.full(te.sum(), k)), ('thr', np.full(te.sum(), thr))]:
            O[key].append(val)
        print(f'  fold {k + 1}/{K_FOLDS}: 학습 {len(fit_ev)}·보정 {n_cal}·평가 {len(te_ev)} 이벤트 · {te.sum()}편 [{time.time() - t0:.0f}s]')
    for k in O: O[k] = np.concatenate(O[k])
    pmf, binp, bb = O['pmf'], O['bin'], O['bins']

    rps = round(rps_censored(pmf, bb), 4)
    F_ = np.cumsum(pmf[:, :K_DAYS], 1)
    calib = {}
    for kd in range(K_DAYS):
        pred = F_[:, kd]; emp = (bb <= kd).astype(float)
        calib[f'P(t<={kd}d)'] = {'pred': round(float(pred.mean()), 3), 'emp': round(float(emp.mean()), 3)}
    ev_mask = bb <= K_DAYS - 1
    auc2_pmf = auc_of(F_[:, 2], (bb <= 2).astype(float))

    m2 = O['b2m']
    acc_bin = float(((binp[m2] >= O['thr'][m2]) == (O['b2l'][m2] == 1)).mean())
    auc_bin = auc_of(binp[m2], O['b2l'][m2])
    acc_pmf2 = float((((1 - F_[:, 2][m2]) < 0.5) == (O['b2l'][m2] == 1)).mean())

    pr = binp[m2] >= O['thr'][m2]; yy = O['b2l'][m2] == 1
    Se = float((pr & yy).sum() / max(yy.sum(), 1)); Sp = float(((~pr) & (~yy)).sum() / max((~yy).sum(), 1))
    LRp = round(Se / max(1 - Sp, 1e-9), 2); LRn = round((1 - Se) / max(Sp, 1e-9), 3)

    evs = sorted(set(O['eid'][m2].tolist()))
    accs = []
    r3 = np.random.RandomState(1)
    for _ in range(300):
        pick = r3.choice(evs, len(evs), replace=True)
        mm = np.isin(O['eid'], pick) & m2
        if mm.sum() > 50: accs.append(((binp[mm] >= O['thr'][mm]) == (O['b2l'][mm] == 1)).mean())
    acc_ci = [round(float(np.percentile(accs, 5)), 3), round(float(np.percentile(accs, 95)), 3)] if accs else None

    print(f'\n===== 모델②′ v1 (이벤트교차 {K_FOLDS}fold · M{ENSEMBLE_M}) =====')
    print(f'  [생존] 절단 RPS {rps} · 2일지평 AUC(PMF) {round(auc2_pmf, 3)}')
    print(f'  [보정곡선] ' + ' · '.join(f'{k}: 예측 {v["pred"]}/실측 {v["emp"]}' for k, v in calib.items()))
    print(f'  [벤치마크(±2일)] 이진 acc {round(acc_bin, 3)} (90%CI {acc_ci}) · AUC {round(auc_bin, 3)} · PMF경로 acc {round(acc_pmf2, 3)}')
    print(f'        (발표 기준선 0.74/0.76 · 2026 SOTA ~0.80-0.82 · 라벨 노이즈 천장 문헌 명시)')
    print(f'  [융합 인터페이스] F1임계 Se {round(Se, 3)} · Sp {round(Sp, 3)} → LR+ {LRp} · LR- {LRn}')
    print(f'        ※ Se/Sp는 C172 모집단 기준 — 터보팬 이식 시 재추정 필요(스펙트럼 편향)')

    np.savez_compressed('preds_ngafid_surv_v1.npz', **O)
    fusion = {'model': 'ngafid-surv-v1', 'horizon_days': K_DAYS,
              'sensitivity': round(Se, 4), 'specificity': round(Sp, 4), 'LR_pos': LRp, 'LR_neg': LRn,
              'f1_threshold_mean': round(float(O['thr'].mean()), 4),
              'usage': 'L1: alert=P(bin)>=thr → Tier B 게이팅 / L2: 사후오즈 = 사전오즈 × LR',
              'caveat': 'C172 일반항공 모집단 — 타 기종 적용 전 Se/Sp 재추정 필수'}
    json.dump({'model': 'ngafid-surv-v1.1', 'protocol': EVENT_GROUPING, 'n_flights': int(len(pmf)), 'n_events': int(len(events)),
               'rps_censored': rps, 'auc_2d_pmf': round(auc2_pmf, 3), 'calibration': calib,
               'benchmark_acc': round(acc_bin, 3), 'benchmark_acc_ci90': acc_ci, 'benchmark_auc': round(auc_bin, 3),
               'acc_pmf_path': round(acc_pmf2, 3), 'fusion_interface': fusion,
               'notes': ['이벤트 그룹 교차(원 논문보다 엄격한 분할) — 발표수치와 직접 비교 시 명시',
                          'after 비행은 5일 내 무사건으로 가정(데이터 라벨 한계, 원 저자 명시)',
                          'same 비행 제외(NGAFID-MC 선례)']},
              open('results_ngafid_surv_v1.json', 'w'), ensure_ascii=False, indent=2)
    json.dump(fusion, open('fusion_interface_model2.json', 'w'), ensure_ascii=False, indent=2)
    print('\n저장: results_ngafid_surv_v1.json · preds_ngafid_surv_v1.npz · fusion_interface_model2.json')
    return acc_bin, auc_bin, rps


In [ ]:
# 데이터 확인
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception: pass
ensure_data()
import pandas as pd
hdr = pd.read_csv(DATA_ROOT + '/flight_header.csv', index_col='Master Index')
bins, is_b, is_a, t, eid, keep, b2l, b2m = build_labels(hdr)
print('전체 비행:', len(hdr), '· same 제외:', (~keep).sum(), '· 이벤트(의사):', len(set(eid.tolist())))
print('before 빈분포(0~4일+오버플로):', __import__('numpy').bincount(bins[keep & is_b], minlength=K_DAYS+1).tolist())
print('after(절단):', (keep & is_a).sum(), '· ±2일 벤치마크 서브셋:', (keep & b2m).sum())
print('→ 3번 학습 셀로. (최초엔 전처리 캐시 생성에 20~40분)')


In [ ]:
# 학습 및 평가
import shutil
acc, auc, rps = run2()
for fn in ('results_ngafid_surv_v1.json', 'preds_ngafid_surv_v1.npz', 'fusion_interface_model2.json'):
    try:
        shutil.copy(fn, '/content/drive/MyDrive/제주항공/'); print('-> Drive:', fn)
    except Exception as e:
        print('Drive 복사 건너뜀(%s):' % fn, e)


In [ ]:
# 모델 1 연동 인터페이스 확인
import json, numpy as np
F = json.load(open('fusion_interface_model2.json'))
z = np.load('preds_ngafid_surv_v1.npz')
print('융합 인터페이스:', {k: F[k] for k in ('sensitivity', 'specificity', 'LR_pos', 'LR_neg')})
Fc = np.cumsum(z['pmf'][:, :K_DAYS], 1)
print('\n예시 5편의 S₂(k) = P(k일 내 정비 불필요):')
for i in np.random.RandomState(0).choice(len(Fc), 5, replace=False):
    print('  비행 %6d: %s (실제 빈 %d)' % (i, np.round(1 - Fc[i], 2).tolist(), z['bins'][i]))
print('\nL2 베이즈 갱신 예시 (통합설계서 수식): 사전 π₀=0.2일 때')
for alert, LR in [('경보', F['LR_pos']), ('무경보', F['LR_neg'])]:
    O0 = 0.2 / 0.8; O = O0 * LR; pi = O / (1 + O)
    print('  모델② %s → 열화 사후확률 π = %.3f → 모델① 혼합/게이팅에 입력' % (alert, pi))
print('\nL1 게이팅: 경보 시 모델①의 Tier B(유닛수준 보수 행동), 무경보 시 Tier A — v14.2 셀④와 연결')
